Import Statements

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

Input data and LSTM parameters/hyperparameters

In [13]:
#Get Dataset
dataset = pd.read_csv('../data/processed/polymarket_cleaned_features.csv')
#Data preprocessing
dataset['Date'] = pd.to_datetime(dataset['Date'])
dataset = dataset.sort_values(['market_id', 'Date']).reset_index(drop=True)
#Data feature headers
features = [
    'price', 'vol_quote', 'final_settlement', 'Target_Deviation', 
    'price_lag_1d', 'price_lag_3d', 'rolling_volatility_7d', 'day_of_week_sin', 
    'day_of_week_cos', 'price_std', 'vol_quote_std', 'price_lag_1d_std', 
    'price_lag_3d_std', 'rolling_volatility_7d_std', 'price_robust', 'vol_quote_robust', 
    'price_lag_1d_robust', 'price_lag_3d_robust', 'rolling_volatility_7d_robust'
]
data = dataset[features].values

#Split data into training and testing
train_size = int(len(dataset) * 0.8)
train_data = dataset.iloc[:train_size]
test_data = dataset.iloc[train_size:]

#Normalize data
normalize = MinMaxScaler()
normalize.fit(train_data[features])

def scale_data(data, scaler, features):
    scaled_values = scaler.transform(data[features])
    new_data = data.copy()
    new_data[features] = scaled_values
    return new_data

train_scaled_data = scale_data(train_data, normalize, features)
test_scaled_data = scale_data(test_data, normalize, features)

#LSTM model hyperparameters
input_size = len(features) #Number of used input features
hidden_size = 256
num_layers = 3
sequence_length = 5 #Shortest sequence length for polymarket data is 6 days, longest is 26 days
learning_rate = 0.0001
epochs = 200
output_size = len(features) #Number of output features
num_forecasted_days = 7 #Number of days to forecast prices

Create LSTM model

In [11]:
#Create the model
class LSTM(nn.Module):
    #Initialize the LSTM model
    def __init__(self, input_size=6, hidden_size=64, num_layers=2, output_size=1):
        super(LSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.hidden_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, states = self.lstm(x)
        return self.hidden_layer(output[:, -1, :])

Train LSTM model on data

In [14]:
lstm = LSTM(input_size, hidden_size, num_layers, output_size=output_size)
criterion = nn.MSELoss()
optimizer = optim.Adam(lstm.parameters(), lr=learning_rate)

def create_grouped_sequences(data, feature_cols, sequence_length):
    xs = []
    ys = []
    data = data.sort_values(['market_id', 'Date'])
    for market_id, group in data.groupby('market_id'):
        group_data = group[feature_cols].values
        if len(group_data) > sequence_length:
            for i in range(len(group_data) - sequence_length):
                xs.append(group_data[i : i + sequence_length])
                ys.append(group_data[i + sequence_length])
    if len(xs) == 0:
        raise ValueError(f"No sequences were created! Check if your markets have at least {sequence_length + 1} rows.")
    xs_np = np.array(xs)
    ys_np = np.array(ys)
    return torch.tensor(xs_np, dtype=torch.float32), torch.tensor(ys_np, dtype=torch.float32)

X_train, y_train = create_grouped_sequences(train_scaled_data, features, sequence_length)
X_test, y_test = create_grouped_sequences(test_scaled_data, features, sequence_length)

train_dataset = TensorDataset(X_train, y_train)
loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

lstm.train()
for epoch in range(epochs):
    lstm.train()
    train_loss = 0
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        outputs = lstm(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    lstm.eval() 
    with torch.no_grad():
        test_outputs = lstm(X_test)
        test_loss = criterion(test_outputs, y_test)
    if (epoch + 1) % 10 == 0:
        avg_train_loss = train_loss / len(loader)
        print(f'Epoch [{epoch+1}/{epochs}] - Train Loss: {avg_train_loss:.5f}, Test Loss: {test_loss.item():.5f}')

Epoch [10/200] - Train Loss: 0.01809, Test Loss: 0.01983
Epoch [20/200] - Train Loss: 0.00304, Test Loss: 0.00228
Epoch [30/200] - Train Loss: 0.00180, Test Loss: 0.00203
Epoch [40/200] - Train Loss: 0.00150, Test Loss: 0.00199
Epoch [50/200] - Train Loss: 0.00135, Test Loss: 0.00176
Epoch [60/200] - Train Loss: 0.00123, Test Loss: 0.00157
Epoch [70/200] - Train Loss: 0.00111, Test Loss: 0.00152
Epoch [80/200] - Train Loss: 0.00103, Test Loss: 0.00165
Epoch [90/200] - Train Loss: 0.00102, Test Loss: 0.00148
Epoch [100/200] - Train Loss: 0.00096, Test Loss: 0.00136
Epoch [110/200] - Train Loss: 0.00087, Test Loss: 0.00125
Epoch [120/200] - Train Loss: 0.00092, Test Loss: 0.00121
Epoch [130/200] - Train Loss: 0.00078, Test Loss: 0.00115
Epoch [140/200] - Train Loss: 0.00076, Test Loss: 0.00114
Epoch [150/200] - Train Loss: 0.00071, Test Loss: 0.00106
Epoch [160/200] - Train Loss: 0.00065, Test Loss: 0.00103
Epoch [170/200] - Train Loss: 0.00060, Test Loss: 0.00093
Epoch [180/200] - Train

Output data/results/conclusions

In [17]:
def prediction(model, input_sequence, normalize):
    model.eval()
    with torch.no_grad():
        scaled_prediction = model(input_sequence).cpu().numpy() 
        inverse_array = normalize.inverse_transform(scaled_prediction)
        return inverse_array[:, 0]

def forecast_future(model, last_sequence, scaler, days_to_predict=7):
    model.eval()
    current_sequence = last_sequence.clone()
    predictions_scaled = []
    with torch.no_grad():
        for _ in range(days_to_predict):
            prediction = model(current_sequence)
            predictions_scaled.append(prediction.squeeze().cpu().numpy())
            current_sequence = torch.cat((current_sequence[:, 1:, :], prediction.unsqueeze(1)), dim=1)
    
    full_prediction_array = np.array(predictions_scaled)
    inverse_array = scaler.inverse_transform(full_prediction_array)
    return inverse_array[:, 0]

full_scaled_data = scale_data(dataset, normalize, features)
unique_all_markets = dataset['market_id'].unique()
all_forecasts = []

print(f"Generating forecasts for {len(unique_all_markets)} markets...")

for target_id in unique_all_markets:
    market_group = full_scaled_data[full_scaled_data['market_id'] == target_id]
    if len(market_group) >= sequence_length:
        recent_data_values = market_group[features].tail(sequence_length).values
        latest_data_sequence = torch.tensor(np.array(recent_data_values), dtype=torch.float32).unsqueeze(0)
        future_prices = forecast_future(lstm, latest_data_sequence, normalize, num_forecasted_days)
        all_forecasts.append([target_id] + list(future_prices))

forecast_cols = ['market_id'] + [f'Day_{i+1}' for i in range(num_forecasted_days)]
forecast_df = pd.DataFrame(all_forecasts, columns=forecast_cols)
print("\n--- Final Forecast Results ---")
print(forecast_df.head(10))

forecast_df['Total_Change'] = forecast_df['Day_7'] - forecast_df['Day_1']

top_movers = forecast_df.sort_values(by='Total_Change', ascending=False)

print("\n--- Top 5 Predicted Price Increases ---")
print(top_movers[['market_id', 'Day_1', 'Day_7', 'Total_Change']].head(5))


def forecast_external_market(model, new_market_data, scaler, features):
    """
    new_market_data: A DataFrame with exactly 5 rows and the 19 feature columns
    """
    model.eval()
    scaled_values = scaler.transform(new_market_data[features])
    input_tensor = torch.tensor(np.array(scaled_values), dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        forecast = forecast_future(model, input_tensor, scaler, days_to_predict=7)
    return forecast

# EXAMPLE USAGE:
# new_data = pd.read_csv('new_market_last_5_days.csv')
# prediction = forecast_external_market(lstm, new_data, normalize, features)
# print(f"Predicted Future: {prediction}")

Generating forecasts for 117 markets...

--- Final Forecast Results ---
                                           market_id     Day_1     Day_2  \
0  0x04a1462e83c415194ac700181bd97f97db149fdcfa4f...  0.569220  0.566938   
1  0x05cb859372049ea33d8cd04aa968b258aa2f28521554...  1.001006  1.006168   
2  0x069274e4970955a64b5fe0656875833610665d54912e...  1.003157  1.008215   
3  0x072576ec724665ab902610fbd23f62813afe96a36674...  0.974985  0.982740   
4  0x0a53315492bf8b043307d67133fadba455f10a097201...  1.001536  1.006689   
5  0x0cd947fe9f2065651071f520e0938e0bb3db009ed8fe...  1.004105  1.009006   
6  0x11097c3daad854876a8d9ef2abdd43dc327b5be1cb5f...  1.004662  1.009570   
7  0x12a561b723e051783fb5f5a67fbc9764986ae2996ca7...  0.998730  1.003890   
8  0x184228c1bfc9502d38f6f9984667f3da7b2f955616f1...  0.997659  1.003231   
9  0x19848149723bbd00545b98f45aa239864d8afd2194fe...  0.502182  0.500377   

      Day_3     Day_4     Day_5     Day_6     Day_7  
0  0.570471  0.568204  0.565418  0.56